# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [ ]:
aviation_df = pd.read_csv(
    'aviationData/AviationData.csv',
    #Fixes the error => UnicodeDecodeError: 'utf-8' codec can't decode byte ... 
    encoding='cp1252', 
    #Helps pandas process data and infer column types when there are different value types
    low_memory=False
)


In [3]:
aviation_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [4]:
aviation_df.dtypes

Event.Id                      str
Investigation.Type            str
Accident.Number               str
Event.Date                    str
Location                      str
Country                       str
Latitude                      str
Longitude                     str
Airport.Code                  str
Airport.Name                  str
Injury.Severity               str
Aircraft.damage               str
Aircraft.Category             str
Registration.Number           str
Make                          str
Model                         str
Amateur.Built                 str
Number.of.Engines         float64
Engine.Type                   str
FAR.Description               str
Schedule                      str
Purpose.of.flight             str
Air.carrier                   str
Total.Fatal.Injuries      float64
Total.Serious.Injuries    float64
Total.Minor.Injuries      float64
Total.Uninjured           float64
Weather.Condition             str
Broad.phase.of.flight         str
Report.Status 

In [5]:
aviation_df.isna().sum()

Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: i

In [6]:
aviation_df.columns

Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date'],
      dtype='str')

In [7]:
aviation_df.loc[0:5, ['Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Publication.Date']]

,Aircraft.Category,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,FAR.Description,Publication.Date
0,NaN,NC6404,Stinson,108-3,No,1.0,Reciprocating,NaN,NaN
1,NaN,N5069P,Piper,PA24-180,No,1.0,Reciprocating,NaN,19-09-1996
2,NaN,N5142R,Cessna,172M,No,1.0,Reciprocating,NaN,26-02-2007
3,NaN,N1168J,Rockwell,112,No,1.0,Reciprocating,NaN,12-09-2000
4,NaN,N15NY,Cessna,501,No,NaN,NaN,NaN,16-04-1980
5,Airplane,CF-TLU,Mcdonnell Douglas,DC9,No,2.0,Turbo Fan,Part 129: Foreign,19-09-2017


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [8]:
aviation_df_relevant = aviation_df.copy()
aviation_df_relevant = aviation_df_relevant.drop(columns=['Latitude', 'Longitude', 'Airport.Name'])
aviation_df_relevant.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Airport.Code,Injury.Severity,Aircraft.damage,Aircraft.Category,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,Fatal(2),Destroyed,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,Fatal(4),Destroyed,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,NaN,Fatal(3),Destroyed,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,Fatal(2),Destroyed,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,Fatal(1),Destroyed,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [9]:
aviation_df_relevant["Event.Date"].dtype

<StringDtype(storage='python', na_value=nan)>

In [ ]:
#Change Event.Date to Datetime type
aviation_df_relevant['Event.Date'] = pd.to_datetime(
    aviation_df_relevant['Event.Date'],
    errors='coerce'
)

In [ ]:
#Filter dataset for records starting from 1983
aviation_df_relevant = aviation_df_relevant.loc[
    aviation_df_relevant['Event.Date'].between(
        '1983-01-01',
        '2023-12-31'
    )
]

len(aviation_df_relevant)

85289

In [12]:
print("Earliest event:", aviation_df_relevant['Event.Date'].min())
print("Latest event:", aviation_df_relevant['Event.Date'].max())

Earliest event: 1983-01-01 00:00:00
Latest event: 2022-12-29 00:00:00


In [13]:
aviation_df_relevant.info()

<class 'pandas.DataFrame'>
RangeIndex: 85289 entries, 3600 to 88888
Data columns (total 28 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                85289 non-null  str           
 1   Investigation.Type      85289 non-null  str           
 2   Accident.Number         85289 non-null  str           
 3   Event.Date              85289 non-null  datetime64[us]
 4   Location                85237 non-null  str           
 5   Country                 85073 non-null  str           
 6   Airport.Code            48435 non-null  str           
 7   Injury.Severity         84289 non-null  str           
 8   Aircraft.damage         82151 non-null  str           
 9   Aircraft.Category       28723 non-null  str           
 10  Registration.Number     83907 non-null  str           
 11  Make                    85232 non-null  str           
 12  Model                   85211 non-null  str           

In [14]:
aviation_df_relevant.describe()

,Event.Date,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,85289,79206.000000,73913.000000,72808.000000,73384.000000,79386.000000
mean,2000-06-09 10:36:58.443175,1.147060,0.657530,0.283595,0.360828,5.461051
min,1983-01-01 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1990-02-18 00:00:00,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1999-05-30 00:00:00,1.000000,0.000000,0.000000,0.000000,1.000000
75%,2009-12-21 00:00:00,1.000000,0.000000,0.000000,0.000000,2.000000
max,2022-12-29 00:00:00,8.000000,349.000000,161.000000,380.000000,699.000000
std,NaN,0.446932,5.576903,1.574753,2.275808,28.412117


In [16]:
aviation_df_relevant.describe(include='str')

,Event.Id,Investigation.Type,Accident.Number,Location,Country,Airport.Code,Injury.Severity,Aircraft.damage,Aircraft.Category,Registration.Number,...,Amateur.Built,Engine.Type,FAR.Description,Schedule,Purpose.of.flight,Air.carrier,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
count,85289,85289,85289,85237,85073,48435,84289,82151,28723,83907,...,85188,78194,28459,12045,79102,16413,80798,58126,78905,71520
unique,84397,2,85263,27008,219,10141,107,4,15,76057,...,2,12,30,3,26,13413,4,12,17074,2554
top,20001214X45071,Accident,ERA22LA103,"ANCHORAGE, AK",United States,NONE,Non-Fatal,Substantial,Airplane,NONE,...,No,Reciprocating,091,NSCH,Personal,Pilot,VMC,Landing,Probable Cause,25-09-2020
freq,3,81534,2,421,78677,1488,64537,61775,24441,341,...,76960,66294,18221,4315,47505,258,74094,14486,58154,17019


In [ ]:
#Inspect major columns
columns_to_inspect = [
    'Aircraft.damage',
    'Amateur.Built',
    'Aircraft.Category',
    'Weather.Condition',
    'Broad.phase.of.flight',
    'Purpose.of.flight',
    'Injury.Severity'
]

for column in columns_to_inspect:
    print(f'\n--- {column} ---')
    print(
        aviation_df_relevant[column]
        .value_counts(dropna=False)
    )



--- Aircraft.damage ---
Aircraft.damage
Substantial    61775
Destroyed      17575
NaN             3138
Minor           2682
Unknown          119
Name: count, dtype: int64

--- Amateur.Built ---
Amateur.Built
No     76960
Yes     8228
NaN      101
Name: count, dtype: int64

--- Aircraft.Category ---
Aircraft.Category
NaN                  56566
Airplane             24441
Helicopter            3151
Glider                 456
Balloon                201
Weight-Shift           161
Gyrocraft              158
Powered Parachute       91
Ultralight              29
Unknown                 13
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64

--- Weather.Condition ---
Weather.Condition
VMC    74094
IMC     5649
NaN     4491
UNK      793
Unk      262
Name: count, dtype: int64

--- Broad.phase.of.flight ---
Broad.phase.of.flight
NaN            27163
Landing      

In [18]:
#Removing any whitespace
aviation_df_relevant['Aircraft.damage'] = (
    aviation_df_relevant['Aircraft.damage'].str.strip()
)

aviation_df_relevant['Aircraft.damage'].value_counts(
    dropna=False
)

Aircraft.damage
Substantial    61775
Destroyed      17575
NaN             3138
Minor           2682
Unknown          119
Name: count, dtype: int64

In [19]:
#Removing any whitespace and standardizing similar values
aviation_df_relevant['Aircraft.Category'] = (
    aviation_df_relevant['Aircraft.Category']
    .str.strip()
    .replace({
        'WSFT': 'Weight-Shift',
        'ULTR': 'Ultralight',
        'UNK': 'Unknown'
    })
)

aviation_df_relevant['Aircraft.Category'].value_counts(
    dropna=False
)

Aircraft.Category
NaN                  56566
Airplane             24441
Helicopter            3151
Glider                 456
Balloon                201
Weight-Shift           170
Gyrocraft              158
Powered Parachute       91
Ultralight              30
Unknown                 15
Powered-Lift             5
Blimp                    4
Rocket                   1
Name: count, dtype: int64

In [20]:
#Removing any whitespace and standardizing similar values
aviation_df_relevant['Weather.Condition'] = (
    aviation_df_relevant['Weather.Condition']
    .str.strip()
    .replace({
        'Unk': 'Unknown',
        'UNK': 'Unknown'
    })
)

aviation_df_relevant['Weather.Condition'].value_counts(
    dropna=False
)

Weather.Condition
VMC        74094
IMC         5649
NaN         4491
Unknown     1055
Name: count, dtype: int64

In [21]:
#Inspecting the main numeric columns(i.e their types and null values)
numeric_columns = [
    'Number.of.Engines',
    'Total.Fatal.Injuries',
    'Total.Serious.Injuries',
    'Total.Minor.Injuries',
    'Total.Uninjured'
]

# Check data types
print(aviation_df_relevant[numeric_columns].dtypes)

# Count missing values
print(aviation_df_relevant[numeric_columns].isnull().sum())

Number.of.Engines         float64
Total.Fatal.Injuries      float64
Total.Serious.Injuries    float64
Total.Minor.Injuries      float64
Total.Uninjured           float64
dtype: object
Number.of.Engines          6083
Total.Fatal.Injuries      11376
Total.Serious.Injuries    12481
Total.Minor.Injuries      11905
Total.Uninjured            5903
dtype: int64


In [22]:
#Filter the dataset to only include aircraft that are professionally built
aviation_df_filtered = aviation_df_relevant[aviation_df_relevant['Amateur.Built'] == 'No']

In [23]:
aviation_df_filtered.columns

Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Airport.Code', 'Injury.Severity',
       'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make',
       'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type',
       'FAR.Description', 'Schedule', 'Purpose.of.flight', 'Air.carrier',
       'Total.Fatal.Injuries', 'Total.Serious.Injuries',
       'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition',
       'Broad.phase.of.flight', 'Report.Status', 'Publication.Date'],
      dtype='str')

In [24]:
professional_df = aviation_df_filtered.copy()
professional_df['Destroyed'] = np.select(
    [
        professional_df['Aircraft.damage'].eq('Destroyed'),
        professional_df['Aircraft.damage'].isin(['Substantial', 'Minor'])
    ],
    [1, 0],
    default=np.nan
)

professional_df['Destroyed'].value_counts(dropna=False)

Destroyed
0.0    58283
1.0    15469
NaN     3208
Name: count, dtype: int64

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [25]:
question_2_df = aviation_df_filtered.copy()

Assumption: Missing counts are not assumed to be zero.

In [26]:
#New derived column: Estimated.Occupants
injury_columns = [
    #relevant columns
    'Total.Fatal.Injuries',
    'Total.Serious.Injuries',
    'Total.Minor.Injuries',
    'Total.Uninjured'
]

question_2_df['Estimated.Occupants'] = (
    #Using min_count=4 means all four counts must be present. 
    #Otherwise, the estimated occupant count remains missing.
    question_2_df[injury_columns].sum(axis=1, min_count=4)
)

In [27]:
#New derived column: Total.Serious.Injuries
question_2_df['Fatal.Serious.Injuries'] = (
    #Only add combined count only when both injury counts are available.
    question_2_df[
        ['Total.Fatal.Injuries', 'Total.Serious.Injuries']
    ].sum(axis=1, min_count=2)
)

In [28]:
#New derived column: Fatal.Serious.Injury.Rate
#This rate is more comparable across aircraft sizes than a raw count alone.
#.where() prevents division by zero. When the occupant count is zero or missing, the rate remains missing.
question_2_df['Fatal.Serious.Injury.Rate'] = (
    question_2_df['Fatal.Serious.Injuries']
    / question_2_df['Estimated.Occupants']
).where(
    question_2_df['Estimated.Occupants'] > 0
)

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [29]:
#New derived column: Aircraft.Destroyed
#np.select() as a way to apply multiple if/elif/else conditions to an entire pandas column at once.
#1 means destroyed and 0 means other values
#Unknown or missing damage is left unclassified
question_2_df['Aircraft.Destroyed'] = np.select(
    [
        question_2_df['Aircraft.damage'].eq('Destroyed'),
        question_2_df['Aircraft.damage'].isin(['Substantial', 'Minor'])
    ],
    [1, 0],
    default=np.nan
)

question_2_df['Aircraft.Destroyed'].value_counts(dropna=False)

Aircraft.Destroyed
0.0    58283
1.0    15469
NaN     3208
Name: count, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [30]:
question_2_df['Make'].value_counts().head(50)

Make
Cessna                            20818
Piper                             11260
CESSNA                             4922
Beech                              4052
PIPER                              2839
Bell                               2006
Boeing                             1498
BOEING                             1151
BEECH                              1042
Mooney                             1029
Grumman                             987
Robinson                            914
Bellanca                            819
Hughes                              740
Schweizer                           609
BELL                                588
Air Tractor                         577
Mcdonnell Douglas                   511
Aeronca                             458
Maule                               427
Champion                            413
De Havilland                        365
Aero Commander                      342
Stinson                             330
Aerospatiale                       

Cleaning Tasks:
    Remove any leading or trailing whitespace,
    Use a standardized upper case for all values,
    Replace multiple spaces inside a value with one space,
    Identify any misspellings or the same make with different names.

In [31]:

question_2_df['Make'] = (
    question_2_df['Make']
    .str.strip()
    .str.upper()
    .str.replace(r'\s+', ' ', regex=True)
)
question_2_df['Make'].value_counts().head(50)

Make
CESSNA                            25740
PIPER                             14099
BEECH                              5094
BOEING                             2649
BELL                               2594
MOONEY                             1271
ROBINSON                           1197
GRUMMAN                            1064
BELLANCA                            978
HUGHES                              877
SCHWEIZER                           753
AIR TRACTOR                         673
AERONCA                             607
MCDONNELL DOUGLAS                   593
MAULE                               571
CHAMPION                            504
STINSON                             421
AERO COMMANDER                      411
DE HAVILLAND                        405
LUSCOMBE                            390
NORTH AMERICAN                      370
TAYLORCRAFT                         366
AEROSPATIALE                        360
ROCKWELL                            337
HILLER                             

In [32]:
#Get the top 50 most common makes
top_50_makes = question_2_df['Make'].value_counts().head(50)
top_50_makes

Make
CESSNA                            25740
PIPER                             14099
BEECH                              5094
BOEING                             2649
BELL                               2594
MOONEY                             1271
ROBINSON                           1197
GRUMMAN                            1064
BELLANCA                            978
HUGHES                              877
SCHWEIZER                           753
AIR TRACTOR                         673
AERONCA                             607
MCDONNELL DOUGLAS                   593
MAULE                               571
CHAMPION                            504
STINSON                             421
AERO COMMANDER                      411
DE HAVILLAND                        405
LUSCOMBE                            390
NORTH AMERICAN                      370
TAYLORCRAFT                         366
AEROSPATIALE                        360
ROCKWELL                            337
HILLER                             

In [33]:
#Filter the dataset for records with any of our top 50 makes
question_2a_df = question_2_df[
    question_2_df['Make'].isin(top_50_makes.index)
].copy()

In [ ]:
#Confirm there are no null values in the Make column
question_2a_df['Make'].isna().sum()

np.int64(0)

In [ ]:
#Clean misspellings and same makes with different names
question_2a_df['Make'] = question_2a_df['Make'].replace({
    "ROBINSON HELICOPTER COMPANY": "ROBINSON HELICOPTER",
    "ROBINSON": "ROBINSON HELICOPTER",
    "AIRBUS INDUSTRIE": "AIRBUS",
    "AIR TRACTOR INC": "AIR TRACTOR"   
    })

question_2a_df['Make'].value_counts()

Make
CESSNA                            25740
PIPER                             14099
BEECH                              5094
BOEING                             2649
BELL                               2594
ROBINSON HELICOPTER                1621
MOONEY                             1271
GRUMMAN                            1064
BELLANCA                            978
AIR TRACTOR                         892
HUGHES                              877
SCHWEIZER                           753
AERONCA                             607
MCDONNELL DOUGLAS                   593
MAULE                               571
CHAMPION                            504
AIRBUS                              447
STINSON                             421
AERO COMMANDER                      411
DE HAVILLAND                        405
LUSCOMBE                            390
NORTH AMERICAN                      370
TAYLORCRAFT                         366
AEROSPATIALE                        360
ROCKWELL                           

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [36]:
question_2a_df['Model'].isna().sum()

np.int64(22)

In [37]:
question_2b_df = question_2a_df.dropna(subset=['Model']).copy()

In [38]:
question_2b_df['Model'].isna().sum()

np.int64(0)

In [39]:
#Check whether models are unique to each make
model_make_counts = (
    question_2b_df.groupby('Model')['Make']
    .nunique()
    .sort_values(ascending=False)
)

model_make_counts.head(20)
    

Model
500           7
300           4
400           4
269C          4
G-164A        4
7GCB          3
200           3
7AC           3
269B          3
7ECA          3
AS-350        3
402           3
7KCAB         3
7FC           3
AS350 B2      3
THRUSH S2R    3
AS350-B2      3
AS350         3
AS350B2       3
100           3
Name: Make, dtype: int64

In [40]:
#New derived column: Make_Model(e.g CESSNA_500)
question_2b_df['Make_Model'] = (
    question_2b_df['Make'].astype('string').str.strip()
    + '_'
    + question_2b_df['Model'].astype('string').str.strip()
)

In [41]:
question_2b_df['Make_Model'].head()

3601         CESSNA_182P
3602        CESSNA_182RG
3603         CESSNA_182P
3604    PIPER_PA-28R-200
3605          CESSNA_140
Name: Make_Model, dtype: string

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [42]:
#Checks the value counts for each column
columns_to_clean = [
    'Engine.Type',
    'Purpose.of.flight',
    'Number.of.Engines',
    'Weather.Condition',
    'Broad.phase.of.flight'
]

for column in columns_to_clean:
    print(f"\nValue counts for {column}:")
    print(question_2b_df[column].value_counts(dropna=False))


Value counts for Engine.Type:
Engine.Type
Reciprocating      53330
NaN                 5059
Turbo Shaft         2877
Turbo Prop          2473
Turbo Fan           1919
Unknown             1192
Turbo Jet            458
Geared Turbofan       12
UNK                    1
NONE                   1
Name: count, dtype: int64

Value counts for Purpose.of.flight:
Purpose.of.flight
Personal                     35513
Instructional                 9196
Unknown                       5181
NaN                           4963
Aerial Application            3777
Business                      3265
Positioning                   1332
Other Work Use                1016
Aerial Observation             691
Public Aircraft                622
Ferry                          607
Executive/corporate            380
Skydiving                      165
Flight Test                    146
Banner Tow                      96
External Load                   85
Public Aircraft - Federal       64
Public Aircraft - State        

In [43]:
# Standardize whitespace in categorical columns
categorical_columns = [
    'Engine.Type',
    'Purpose.of.flight',
    'Weather.Condition',
    'Broad.phase.of.flight'
]

for column in categorical_columns:
    question_2b_df[column] = (
        question_2b_df[column]
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
    )

# Standardize known equivalent labels
question_2b_df['Engine.Type'] = (
    question_2b_df['Engine.Type'].replace({
        'UNK': 'Unknown'
    })
)

question_2b_df['Purpose.of.flight'] = (
    question_2b_df['Purpose.of.flight'].replace({
        'Air Race show': 'Air Race/Show',
        'Air Race/show': 'Air Race/Show'
    })
)

In [44]:
question_2b_df.loc[
    question_2b_df['Purpose.of.flight'].isin(
        ['PUBS', 'PUBL', 'ASHO']
    ),
    ['Purpose.of.flight', 'Make', 'Model']
]

,Purpose.of.flight,Make,Model
87792,PUBS,CESSNA,185
87822,ASHO,CESSNA,140
87897,PUBL,BELL,HH-1H
87976,PUBS,CESSNA,T206H
88259,PUBS,PIPER,PA-18-150
88776,ASHO,BELL,P63
88777,ASHO,BOEING,B17


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [45]:
question_2b_df.isna().sum()

Event.Id                         0
Investigation.Type               0
Accident.Number                  0
Event.Date                       0
Location                        44
Country                        181
Airport.Code                 29240
Injury.Severity                841
Aircraft.damage               2418
Aircraft.Category            47067
Registration.Number           1053
Make                             0
Model                            0
Amateur.Built                    0
Number.of.Engines             4404
Engine.Type                   5059
FAR.Description              47251
Schedule                     57233
Purpose.of.flight             4963
Air.carrier                  54961
Total.Fatal.Injuries          8853
Total.Serious.Injuries        9684
Total.Minor.Injuries          9177
Total.Uninjured               4166
Weather.Condition             3521
Broad.phase.of.flight        19185
Report.Status                 4809
Publication.Date             11805
Estimated.Occupants 

In [ ]:
#Create a df showing the percentage and count of missing values per column 
missing_summary = pd.DataFrame({
    'Missing_Count': question_2b_df.isna().sum(),
    'Missing_Percentage': (
        question_2b_df.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    'Missing_Percentage',
    ascending=False
)

missing_summary

,Missing_Count,Missing_Percentage
Schedule,57233,85.01
Air.carrier,54961,81.64
FAR.Description,47251,70.19
Aircraft.Category,47067,69.91
Airport.Code,29240,43.43
Broad.phase.of.flight,19185,28.50
Fatal.Serious.Injury.Rate,11991,17.81
Publication.Date,11805,17.54
Estimated.Occupants,11100,16.49
Fatal.Serious.Injuries,10733,15.94


In [ ]:
#Final dataset after dropping columns with too many NaNs
final_aviation_df = question_2b_df.drop(
    columns=['Schedule', 'Air.carrier', 'FAR.Description', 'Airport.Code']).copy()

In [52]:
final_aviation_df.columns

Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type',
       'Purpose.of.flight', 'Total.Fatal.Injuries', 'Total.Serious.Injuries',
       'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition',
       'Broad.phase.of.flight', 'Report.Status', 'Publication.Date',
       'Estimated.Occupants', 'Fatal.Serious.Injuries',
       'Fatal.Serious.Injury.Rate', 'Aircraft.Destroyed', 'Make_Model'],
      dtype='str')

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [54]:
final_aviation_df.to_csv('aviationData/FinalAviationData.csv', index=False)